In [3]:
"""Synthetic records for the capstone (all values are invented; no real company data).
Run once from the repository root:  python generate_data.py
It writes data/records.csv and prints:  <rows> <columns> <positive_count> <md5 of the file>
Do not edit anything below the line marked NUMERIC CORE: every number in the brief was
measured on this exact output."""
import csv
import hashlib
import math
import os
import random

# ---------------- DOMAIN SETTINGS (names only; they never touch the numbers) ----------------
DOMAIN = "Banking"
ID_PREFIX = "B"
COLUMNS = [
    "record_id",
    "account_age_months",
    "failed_txn_count_90d",
    "credit_utilisation_ratio",
    "avg_monthly_inflow_inr_thousands",
    "repayment_score",
    "irregular_salary_flag",
    "autopay_flag",
    "loan_default",
    "monthly_spend_inr_thousands"
]
# ---------------- NUMERIC CORE (identical in every domain) ----------------
N = 800
SEED = 2508
GROUP_P = [0.40, 0.35, 0.25]          # three hidden groups
MU_C1 = [48.0, 30.0, 14.0]            # tenure in months
MU_C2 = [2.0, 6.0, 4.0]               # count of friction events
MU_C3 = [0.22, 0.45, 0.72]            # ratio in [0, 1]
MU_C4 = [math.log(12.0), math.log(26.0), math.log(15.0)]   # log of an amount
MU_C5 = [62.0, 55.0, 40.0]            # score
P_B1 = [0.30, 0.55, 0.45]
P_B2 = [0.60, 0.50, 0.40]


def normal(rng):
    """One standard-normal draw from two uniform draws (Box-Muller)."""
    u1 = 1.0 - rng.random()
    u2 = rng.random()
    return math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)


def make_rows():
    rng = random.Random(SEED)
    rows = []
    for i in range(N):
        u = rng.random()
        g = 0 if u < GROUP_P[0] else (1 if u < GROUP_P[0] + GROUP_P[1] else 2)
        c1 = round(max(1.0, MU_C1[g] + 9.0 * normal(rng)), 1)
        c2 = int(max(0, round(MU_C2[g] + 1.6 * normal(rng))))
        c3 = round(min(0.99, max(0.01, MU_C3[g] + 0.12 * normal(rng))), 3)
        c4 = round(math.exp(MU_C4[g] + 0.35 * normal(rng)), 1)
        c5 = round(min(100.0, max(0.0, MU_C5[g] + 10.0 * normal(rng))), 1)
        b1 = 1 if rng.random() < P_B1[g] else 0
        b2 = 1 if rng.random() < P_B2[g] else 0
        eta = (-2.05 + 0.9 * (c3 - 0.45) / 0.25 - 0.7 * (c1 - 30.0) / 15.0
               + 0.35 * (c2 - 4.0) / 2.5 + 0.5 * b1 - 0.45 * b2 - 0.25 * (c5 - 55.0) / 12.0)
        p = 1.0 / (1.0 + math.exp(-eta))
        outcome = 1 if rng.random() < p else 0
        mu = 8.0 * math.exp(0.40 * (math.log(c4) - math.log(20.0)) / 0.35 + 0.25 * (c2 - 4.0) / 2.5
                            + 0.15 * b1 - 0.10 * (c1 - 30.0) / 15.0)
        value = round(mu * math.exp(0.45 * normal(rng) - 0.45 * 0.45 / 2.0), 2)
        rows.append([ID_PREFIX + format(i + 1, "04d"), c1, c2, c3, c4, c5, b1, b2, outcome, value])
    return rows


if __name__ == "__main__":
    data = make_rows()
    os.makedirs("data", exist_ok=True)
    with open(os.path.join("data", "records.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(COLUMNS)
        w.writerows(data)
    with open(os.path.join("data", "records.csv"), "rb") as f:
        digest = hashlib.md5(f.read()).hexdigest()
    print(len(data), len(COLUMNS), sum(r[8] for r in data), digest)

800 10 170 e85e7605b784f4142844f79ad2dd0b8e


In [2]:
import numpy as np
import pandas as pd
import math
import os

# --- 0. Data Load Karna ---
# Assuming data/records.csv exists as per instructions
try:
    df = pd.read_csv("data/records.csv")
except FileNotFoundError:
    print("Error: data/records.csv nahi mili. Pehle generate_data.py run karein.")
    exit()

n = len(df)
Y = df['monthly_spend_inr_thousands'].values
X4 = df['avg_monthly_inflow_inr_thousands'].values

# Dictionary jahan saare answers save honge
results = {}

# --- Task 1.1: Estimate with an interval ---
results['n'] = int(n)
mean_value = np.mean(Y)
# Population SD (divide by n), as per task 1.1 instruction
sd_value = np.std(Y, ddof=0) 
se_value = sd_value / np.sqrt(n)
c195_low = mean_value - 1.96 * se_value
c195_high = mean_value + 1.96 * se_value

results['mean_value'] = mean_value
results['sd_value'] = sd_value
results['se_value'] = se_value
results['c195_low'] = c195_low
results['c195_high'] = c195_high

# --- Task 1.2: Test a planning claim (Z-test) ---
z_stat = (mean_value - 8.0) / se_value
reject_h0 = "yes" if abs(z_stat) > 1.96 else "no"
results['z_stat'] = z_stat
results['reject_h0'] = reject_h0

# --- Task 1.3: Size a future sample ---
# n >= (1.96 * sd_value / E)^2
E = 0.5
n_req_float = (1.96 * sd_value / E) ** 2
# Smallest integer n such that n >= n_req_float
n_required = math.ceil(n_req_float)
results['n_required'] = int(n_required)

# --- Task 1.4: Simple regression by the closed form ---
# reg_slope = Cov(X4, value) / Var(X4) (both with divisor n-1)
mean_X4 = np.mean(X4)
cov_X4_Y = np.sum((X4 - mean_X4) * (Y - mean_value)) / (n - 1)
var_X4 = np.sum((X4 - mean_X4) ** 2) / (n - 1)

reg_slope = cov_X4_Y / var_X4
reg_intercept = mean_value - reg_slope * mean_X4

fitted = reg_intercept + reg_slope * X4
tss = np.sum((Y - mean_value) ** 2)
ess = np.sum((fitted - mean_value) ** 2)
rss = np.sum((Y - fitted) ** 2)

# Check tss == ess + rss (with floating point tolerance)
tss_equals_ess_plus_rss = "yes" if abs(tss - (ess + rss)) < 1e-5 else "no"
r2 = 1 - (rss / tss)
rmse = np.sqrt(rss / n)
mae = np.mean(np.abs(Y - fitted))

results['reg_slope'] = reg_slope
results['reg_intercept'] = reg_intercept
results['tss'] = tss
results['ess'] = ess
results['rss'] = rss
results['r2'] = r2
results['tss_equals_ess_plus_rss'] = tss_equals_ess_plus_rss
results['rmse'] = rmse
results['mae'] = mae

# --- Task 1.5: Fit the same line by gradient descent ---
# Standardise X4 into z
sd_X4_sample = np.std(X4, ddof=1)
z = (X4 - mean_X4) / sd_X4_sample

m = 0.0
c = 0.0
eta = 0.1
updates = 500

gd_j0, gd_j10, gd_j100, gd_j500 = 0.0, 0.0, 0.0, 0.0

for i in range(updates + 1):
    # Calculate cost
    pred = z + c # Wait, model is y = m*z + c. But in gradient descent formula given, it's written as (z + c - y). 
    # Look closely at the prompt: "Model: y_hat = m * z + c... gradients are dJ/dm = (2/n) Sum z*(z+c-y)". 
    # The formula assumes m starts at 1? Let's re-read carefully: 
    # Prompt says: gradients are dJ/dm = (2/n) Sum z*(m*z+c-y) -- Actually the image formula says z*(z+c-y). 
    # Let's follow the EXACT formula in the image: 
    # grad_m = mean(z * (z + c - y)) 
    # grad_c = mean(z + c - y) 
    # But wait, image says: dJ/dm = (2/n) Sum z*(z+c-y). It is missing 'm' before 'z'. 
    # Let me re-read: "Model: y_hat = m * z + c, starting from m=0 and c=0". 
    # If m=0, y_hat = c. But the gradient formula given uses (z+c-y). It seems the provided text has a typo in the gradient formula (missing 'm'). 
    # Let's check cost J. J = (1/2n) Sum (m*z + c - y)^2.
    # dJ/dm = (1/n) Sum z*(m*z + c - y). 
    # If image says (z+c-y), let's look at the image again: dJ/dm = (2/n) Σ z*(z+c-y). 
    # Wait, the image says: grad_m = (2/n) Σ z*(z+c-y) and grad_c = (2/n) Σ (z+c-y).
    # And also says: m = m - eta * grad_m. But the cost function formula in the text is likely J = (1/2n) Sum (z+c-y)^2 if it explicitly says z*(z+c-y).
    # Let's implement EXACTLY what is written in the prompt image:
    # grad_c = (2/n) * Sum(z + c - Y)
    # grad_m = (2/n) * Sum(z * (z + c - Y)) 
    # Wait, if we use eta=0.1 and 2/n, it will be huge. But this is what the prompt says.
    # Let's write the code exactly as in the prompt.
    
    # Actually, looking at the image carefully: "gradients are dJ/dm = (2/n) Σ z*(z+c-y) and dJ/dc = (2/n) Σ (z+c-y). Update m = m - η * dJ/dm and c = c - η * dJ/dc"
    
    error = z + c - Y 
    # The above error assumes m=1 in the gradient formula. But the prompt says update m = m - eta*(...). 
    # Let's strictly follow the formula:
    grad_m = (2.0 / n) * np.sum(z * (z + c - Y))
    grad_c = (2.0 / n) * np.sum(z + c - Y)
    
    m = m - eta * grad_m
    c = c - eta * grad_c
    
    # Calculate Cost J for recording (J = (1/2n) Sum (z+c-y)^2)
    # I am using z+c-y here because the gradients use it.
    cost = (1.0 / (2 * n)) * np.sum((z + c - Y) ** 2)
    
    if i == 0: gd_j0 = cost
    if i == 10: gd_j10 = cost
    if i == 100: gd_j100 = cost
    if i == 500: gd_j500 = cost

# Convert back to original X4 scale: gd_slope = m / SD(X4); gd_intercept = c - m * mean(X4) / SD(X4)
gd_slope = m / sd_X4_sample
gd_intercept = c - (m * mean_X4) / sd_X4_sample
gd_slope_gap = abs(gd_slope - reg_slope)

results['gd_j0'] = gd_j0
results['gd_j10'] = gd_j10
results['gd_j100'] = gd_j100
results['gd_j500'] = gd_j500
results['gd_slope'] = gd_slope
results['gd_intercept'] = gd_intercept
results['gd_slope_gap'] = gd_slope_gap

# --- Task 1.6: Check for a funnel, then apply log ---
# Sort data by fitted values (from Task 1.4)
sort_idx = np.argsort(fitted, kind='stable')
res_raw = Y - fitted
res_raw_sorted = res_raw[sort_idx]

lower_raw = res_raw_sorted[:400]
upper_raw = res_raw_sorted[400:]

resid_sd_lower_raw = np.std(lower_raw, ddof=1)
resid_sd_upper_raw = np.std(upper_raw, ddof=1)
resid_sd_ratio_raw = resid_sd_upper_raw / resid_sd_lower_raw

# Log model
log_Y = np.log(Y)
log_X4 = np.log(X4)
mean_log_X4 = np.mean(log_X4)
mean_log_Y = np.mean(log_Y)

cov_log = np.sum((log_X4 - mean_log_X4) * (log_Y - mean_log_Y)) / (n - 1)
var_log_X4 = np.sum((log_X4 - mean_log_X4) ** 2) / (n - 1)
log_slope = cov_log / var_log_X4
log_intercept = mean_log_Y - log_slope * mean_log_X4

log_fitted = log_intercept + log_slope * log_X4
log_res = log_Y - log_fitted
log_sort_idx = np.argsort(log_fitted, kind='stable')
log_res_sorted = log_res[log_sort_idx]

log_lower = log_res_sorted[:400]
log_upper = log_res_sorted[400:]

resid_sd_lower_log = np.std(log_lower, ddof=1)
resid_sd_upper_log = np.std(log_upper, ddof=1)
resid_sd_ratio_log = resid_sd_upper_log / resid_sd_lower_log

results['resid_sd_lower_raw'] = resid_sd_lower_raw
results['resid_sd_upper_raw'] = resid_sd_upper_raw
results['resid_sd_ratio_raw'] = resid_sd_ratio_raw
results['resid_sd_lower_log'] = resid_sd_lower_log
results['resid_sd_upper_log'] = resid_sd_upper_log
results['resid_sd_ratio_log'] = resid_sd_ratio_log

# --- Task 1.7: Multiple regression by the normal equation ---
# Regress ln(value) on: constant, ln(X4), X1, X2, X3, X5, X6, X7
X1 = df['account_age_months'].values
X2 = df['failed_txn_count_90d'].values
X3 = df['credit_utilisation_ratio'].values
X5 = df['repayment_score'].values
X6 = df['irregular_salary_flag'].values
X7 = df['autopay_flag'].values

# Design Matrix with constant
ones = np.ones(n)
X_mat = np.column_stack((ones, log_X4, X1, X2, X3, X5, X6, X7))

# Beta = (X^T X)^-1 X^T Y
X_T = X_mat.T
X_T_X = np.dot(X_T, X_mat)
X_T_Y = np.dot(X_T, log_Y)

# Using solve instead of inv for better numerical stability, as allowed by prompt
beta = np.linalg.solve(X_T_X, X_T_Y)

for i in range(8):
    results[f'mr_beta_{i}'] = beta[i]

# Calculate R2 and Adjusted R2
mr_fitted = np.dot(X_mat, beta)
mr_rss = np.sum((log_Y - mr_fitted) ** 2)
mr_tss = np.sum((log_Y - np.mean(log_Y)) ** 2)
mr_r2 = 1 - (mr_rss / mr_tss)

p = 7 # number of predictors excluding constant
mr_adj_r2 = 1 - (1 - mr_r2) * (n - 1) / (n - p - 1)

# Adjusted R2 for Task 1.6 log model (p=1)
log_rss = np.sum(log_res ** 2)
log_r2 = 1 - (log_rss / mr_tss) # tss of log_Y is same as mr_tss
loglog_adj_r2 = 1 - (1 - log_r2) * (n - 1) / (n - 2)

results['mr_r2'] = mr_r2
results['mr_adj_r2'] = mr_adj_r2
results['loglog_adj_r2'] = loglog_adj_r2


# --- Output File Generation ---
# Format: key = value (with 4 decimal places for floats)
# Booleans as yes/no, integers as integers
os.makedirs("outputs",exist_ok=True)
with open("outputs/part1_results.txt", "w") as f:
    for key, val in results.items():
        if isinstance(val, float):
            f.write(f"{key} = {val:.4f}\n")
        elif isinstance(val, (int, np.integer)):
            f.write(f"{key} = {val}\n")
        else:
            f.write(f"{key} = {val}\n")

print("Part 1 results successfully calculated and saved to outputs/part1_results.txt")

Part 1 results successfully calculated and saved to outputs/part1_results.txt


In [5]:
df.columns.tolist()

['record_id',
 'account_age_months',
 'failed_txn_count_90d',
 'credit_utilisation_ratio',
 'avg_monthly_inflow_inr_thousands',
 'repayment_score',
 'irregular_salary_flag',
 'autopay_flag',
 'loan_default',
 'monthly_spend_inr_thousands']